# Stable Diffusion 2.1 generation at 50 denoising steps

This notebook reuses the full U-Net checkpoint family trained by `02_SD21_Filtered_100steps.ipynb`, evaluates every candidate under the 50-step sampling protocol, and independently selects the lowest-mean-FID candidate on the validation partition. It then generates the unfiltered class-conditional pools. Test data remain reserved for final classifier evaluation.

Checkpoint files remain under `experiments/diffusers/02_sd21_filtered_100steps/model/`; G01 writes only its own evaluation images, final images, caches, metrics, manifests, and sustainability logs under `experiments/diffusers/01_sd21_baseline_50steps/`. Sharing the checkpoint family does not, by itself, guarantee that G01 and G02 select the same checkpoint: their validation images are sampled with different denoising budgets. A 50-versus-100 comparison may therefore be called a sampling-step-only contrast only after the saved generation records and checkpoint content establish identical selected weights. Without that verification, it is a direct, size-controlled descriptive comparison of two independently validation-selected pipelines.


## 1. Initialize dependencies, execution policy, and experiment configuration

The setup cells locate the repository, import the runtime dependencies, and install selected missing Python packages; PyTorch itself must already be installed. They verify the pinned local Diffusers revision, ensure its editable installation, import the shared SD2.1 asset utilities, and configure class prompts, image counts, seeds, resolution, guidance scale, and 50 inference steps. Multi-GPU generation is delegated to the shared scheduler with automatic device discovery.

Execution intent is one boolean per phase. `RUN_TRAINING_PHASE` stays `False` because this notebook reuses notebook 02, and `RUN_FILTER_PHASE` stays `False` because G01 publishes an unfiltered pool; generation and evaluation run only when the operator sets their flag. An ordinary Run All therefore retrains and regenerates nothing. Dataset, archive, cache, source-checkpoint, and final-output paths are then created or resolved.

**Inputs:** the repository utilities, a compatible PyTorch/CUDA environment, processed and augmented data, the shared SD2.1 base, and notebook 02 checkpoints. **Outputs:** imported runtime objects, the declared phase flags, experiment configuration, and initialized directories. Invalid repository paths, a missing PyTorch installation, a mismatched Diffusers revision, or an invalid class label cause explicit failures. This section fixes the ablation protocol before any evaluation result is read.


In [ ]:
# === Unified notebooks/ bootstrap ===
# Works from the project root and every subdirectory under notebooks/.
import sys as _sys
from pathlib import Path as _Path

def _find_mammo_root():
    for _candidate in [_Path.cwd().resolve(), *_Path.cwd().resolve().parents]:
        if _candidate.name == "MammoDiffusion":
            return _candidate
        if (_candidate / "data").is_dir() and (_candidate / "notebooks").is_dir():
            return _candidate
    raise FileNotFoundError("MammoDiffusion root not found from " + str(_Path.cwd()))

PROJECT_ROOT = _find_mammo_root()
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
UTILITY_DIR = NOTEBOOKS_DIR / "utility"
for _path in (str(UTILITY_DIR), str(NOTEBOOKS_DIR)):
    if _path not in _sys.path:
        _sys.path.insert(0, _path)

# === End unified bootstrap ===

# Dependency bootstrap and imports
from pathlib import Path
from tempfile import TemporaryDirectory
from datetime import datetime
from contextlib import contextmanager
import gc
import hashlib
import importlib
import importlib.util
import json
import os
import re
import shutil
import subprocess
import sys
import zipfile

PROJECT_NAME = "MammoDiffusion"
PROJECT_ROOT_OVERRIDE = None  # Colab example: "/content/drive/MyDrive/MammoDiffusion"
EXPERIMENT_NAME = "diffusers/01_sd21_baseline_50steps"

def find_project_root(project_name=PROJECT_NAME, override=PROJECT_ROOT_OVERRIDE):
    if override is not None:
        root = Path(override).expanduser().resolve()
        if not root.is_dir():
            raise FileNotFoundError(f"PROJECT_ROOT_OVERRIDE does not exist: {root}")
        return root

    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        has_notebooks = (candidate / "notebooks").is_dir() or (candidate / "notebooks").is_dir()
        if candidate.name == project_name or ((candidate / "data").is_dir() and has_notebooks) or ((candidate / ".git").exists() and has_notebooks):
            return candidate

    fallback_candidates = [
        cwd / project_name,
        Path("/content") / project_name,
        Path("/content/drive/MyDrive") / project_name,
        Path.home() / project_name,
        Path.home() / "Progetto" / project_name,
    ]
    for candidate in fallback_candidates:
        if candidate.is_dir():
            return candidate.resolve()

    raise FileNotFoundError(
        "MammoDiffusion root not found. Run the notebook from the repository "
        "or set PROJECT_ROOT_OVERRIDE."
    )

PROJECT_ROOT = find_project_root()
NOTEBOOKS_DIR = PROJECT_ROOT / ("notebooks" if (PROJECT_ROOT / "notebooks").is_dir() else "notebooks")
UTILITY_DIR = NOTEBOOKS_DIR / "utility" if (NOTEBOOKS_DIR / "utility").is_dir() else NOTEBOOKS_DIR
for _path in (UTILITY_DIR, NOTEBOOKS_DIR):
    if str(_path) not in sys.path:
        sys.path.insert(0, str(_path))

EXPERIMENTS_DIR = PROJECT_ROOT / "experiments"
EXPERIMENT_DIR = EXPERIMENTS_DIR / EXPERIMENT_NAME
from shared_diffusers_assets import (DIFFUSERS_REVISION, SHARED_DIFFUSERS_REPO_DIR, SHARED_SD21_BASE_DIR, ensure_shared_diffusers_repo, ensure_diffusers_editable_install, shared_diffusers_train_script, verify_diffusers_revision)
DIFFUSERS_REPO_DIR = SHARED_DIFFUSERS_REPO_DIR
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

AUTO_INSTALL_PACKAGES = {
    "gdown": "gdown",
    "matplotlib": "matplotlib",
    "pandas": "pandas",
    "PIL": "Pillow",
    "tqdm": "tqdm",
    "transformers": "transformers",
    "accelerate": "accelerate",
    "datasets": "datasets",
    "safetensors": "safetensors",
    "huggingface_hub": "huggingface_hub",
    "bitsandbytes": "bitsandbytes",
}
missing_packages = [
    package_name
    for module_name, package_name in AUTO_INSTALL_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]
if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])

missing_required = [
    module_name for module_name in ["torch"] if importlib.util.find_spec(module_name) is None
]
if missing_required:
    raise ImportError(
        "Required dependencies not found: "
        + ", ".join(missing_required)
        + ". Install PyTorch in the environment before running the notebook."
    )

DIFFUSERS_REPO_DIR = ensure_shared_diffusers_repo()
verify_diffusers_revision(DIFFUSERS_REPO_DIR)
ensure_diffusers_editable_install(DIFFUSERS_REPO_DIR)
TRAIN_SCRIPT = shared_diffusers_train_script(lora=False)
importlib.invalidate_caches()
# Reload the utility because an active kernel may have cached an
# earlier prepare_sd_manifest signature whose third argument was required.
import parallel_generation_utils as _parallel_generation_utils
_parallel_generation_utils = importlib.reload(_parallel_generation_utils)

for name in [
    "HF_HUB_VERBOSITY",
    "TRANSFORMERS_VERBOSITY",
    "DIFFUSERS_VERBOSITY",
    "ACCELERATE_LOG_LEVEL",
]:
    os.environ[name] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONWARNINGS"] = "ignore"

import gdown
import matplotlib.pyplot as plt
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from diffusers import StableDiffusionPipeline, UNet2DConditionModel
from eco_tracker import measure_sustainability
from generative_evaluator import GenerativeEvaluator

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("PROJECT_ROOT:", PROJECT_ROOT)
# Multi-GPU generation is used only by evaluation and final-dataset creation, never training.
PARALLEL_GENERATION = True
GENERATION_GPU_DEVICES = "auto"
GENERATION_MAX_WORKERS = None
GENERATION_SCHEDULER = "dynamic_reservations"
GENERATION_RESERVATION_SIZE = 4
EVALUATION_GENERATION_SCHEDULER = "auto"
SD_CHECKPOINT_TYPE = "full_unet"

def _sd_parallel_generate_checkpoint_jobs(checkpoints, base_model_dir, eval_dir, negative_prompt, positive_prompt, n_gen, inference_steps, guidance_scale, seed, resolution):
    from parallel_generation_utils import SD_SEED_OFFSETS, missing_named_png_indices, prepare_sd_manifest, run_sd_generation_jobs
    jobs = []
    for step, checkpoint_path in checkpoints:
        requests = []
        for class_name, prompt in (("negative", negative_prompt), ("positive", positive_prompt)):
            class_offset = SD_SEED_OFFSETS[f"evaluation:{class_name}"]
            out_dir = Path(eval_dir) / Path(checkpoint_path).name / class_name
            request = {"name": class_name, "class_name": class_name, "phase": "evaluation", "prompt": prompt, "out_dir": str(out_dir), "count": n_gen, "seed": int(seed), "class_offset": class_offset, "inference_steps": int(inference_steps), "guidance_scale": float(guidance_scale), "resolution": int(resolution), "checkpoint_type": SD_CHECKPOINT_TYPE, "base_model_dir": str(Path(base_model_dir).resolve())}
            missing = missing_named_png_indices(out_dir, n_gen)
            if missing:
                prepare_sd_manifest(request, str(checkpoint_path))
                requests.append({**request, "indices": missing})
        if requests:
            jobs.append({"label": f"checkpoint {step}", "checkpoint_path": str(checkpoint_path), "checkpoint_type": SD_CHECKPOINT_TYPE, "base_model_dir": str(base_model_dir), "requests": requests})
    if not jobs:
        print("Stable Diffusion evaluation: no missing or corrupt image.")
        return []
    return run_sd_generation_jobs(jobs, GENERATION_GPU_DEVICES if PARALLEL_GENERATION else "off", GENERATION_MAX_WORKERS, Path(EXPERIMENT_DIR) / "logs" / "parallel_generation", Path(PROJECT_ROOT), dry_run=False, generation_scheduler=EVALUATION_GENERATION_SCHEDULER, reservation_size=GENERATION_RESERVATION_SIZE)

def _sd_parallel_generate_final(checkpoint_path, base_model_dir, prompt, out_dir, target_total, inference_steps, guidance_scale, seed, resolution, class_name, reused_prefix, n_reused):
    from parallel_generation_utils import SD_SEED_OFFSETS, final_sd_generation_plan, prepare_sd_manifest, run_sd_final_generation
    n_new = int(target_total) - int(n_reused)
    if n_new < 0:
        raise RuntimeError(f"Reused evaluation images ({n_reused}) exceed the target ({target_total}).")
    request = {"name": class_name, "class_name": class_name, "phase": "final_new", "prompt": prompt, "out_dir": str(out_dir), "indices": [], "count": n_new, "seed": int(seed), "class_offset": SD_SEED_OFFSETS[f"final_new:{class_name}"], "inference_steps": int(inference_steps), "guidance_scale": float(guidance_scale), "resolution": int(resolution), "checkpoint_type": SD_CHECKPOINT_TYPE, "base_model_dir": str(Path(base_model_dir).resolve())}
    plan = final_sd_generation_plan(Path(out_dir), target_total, reused_prefix)
    missing = plan["missing_gen_indices"]
    request["indices"] = missing
    if not missing:
        return plan
    prepare_sd_manifest(request, str(checkpoint_path))
    run_sd_final_generation(Path(checkpoint_path), SD_CHECKPOINT_TYPE, Path(base_model_dir), [request], GENERATION_GPU_DEVICES if PARALLEL_GENERATION else "off", GENERATION_MAX_WORKERS, Path(EXPERIMENT_DIR) / "logs" / "parallel_generation", Path(PROJECT_ROOT), dry_run=False, generation_scheduler=GENERATION_SCHEDULER, reservation_size=GENERATION_RESERVATION_SIZE)
    return final_sd_generation_plan(Path(out_dir), target_total, reused_prefix)

In [ ]:
# EXPLICIT_PHASE_FLAGS_V1
# One boolean per phase. Every flag is False, so an ordinary Run All reads the
# artifacts already on disk and reports them: it never retrains and never
# regenerates. To do real work, set the flags for the phases you intend to run
# and execute the notebook top to bottom. Flags are independent -- filtering can
# be redone without regenerating the pool it selects from.
#
#   RUN_TRAINING_PHASE    train the model (hours of GPU)
#   RUN_GENERATION_PHASE  sample a full RAW image pool from the selected checkpoint
#   RUN_EVALUATION_PHASE  score checkpoints and record the selection
#   RUN_FILTER_PHASE      re-run the adaptive filter over the existing RAW pool
#
# Leaving a flag False asserts that the phase's artifact is already complete.
# The cells below check that claim and raise if it does not hold, rather than
# reporting a number they did not verify.
RUN_TRAINING_PHASE = False  # 01 samples the checkpoint trained by 02
RUN_GENERATION_PHASE = False
RUN_EVALUATION_PHASE = False
RUN_FILTER_PHASE = False  # G01 publishes an unfiltered sampling dataset

In [ ]:
# Datasets shared by the project
DATA_DIR = PROJECT_ROOT / "data"
ARCHIVES_DIR = DATA_DIR / "archives"
DATA_PROCESSED_DIR = DATA_DIR / "processed"
DATA_AUG = DATA_DIR / "real_augmented"

PROCESSED_DRIVE_ID = "1qQral_BIBlMl0QN3PllJukdYTOmNGWr3"
AUGMENTED_DRIVE_ID = "1XRc0SxLEPP-zbMJApn4ruaiH8u_rDc-0"
PROCESSED_ZIP_PATH = ARCHIVES_DIR / "processed.zip"
AUGMENTED_ZIP_PATH = ARCHIVES_DIR / "real_augmented.zip"

FORCE_PROCESSED_REDOWNLOAD = False
FORCE_AUGMENTED_REDOWNLOAD = False
FORCE_MODEL_REDOWNLOAD = False

# The experiment and Diffusers repository are defined during initial bootstrap
SD21_MODEL_DRIVE_ID = "10XRn-bxpp7tP6ROWLYpeCNYJZIaHfMUt"
SHARED_PRETRAINED_ROOT = NOTEBOOKS_DIR / "pretrained_model"
PRETRAINED_MODEL_DIR = SHARED_SD21_BASE_DIR
PRETRAINED_MODEL_ZIP_PATH = SHARED_PRETRAINED_ROOT / "archives" / "stable-diffusion-2-1-base.zip"
HF_CACHE_DIR = PROJECT_ROOT / ".cache" / "huggingface" / "01_sd21_baseline_50steps"
SOURCE_EXPERIMENT_DIR = EXPERIMENTS_DIR / "diffusers/02_sd21_filtered_100steps"
SD_OUTPUT_DIR = SOURCE_EXPERIMENT_DIR / "model"

# Final generated images are retained inside the experiment
FINAL_GEN_DIR = EXPERIMENT_DIR / "generated_images" / "final"
FINAL_NEG_DIR = FINAL_GEN_DIR / "negative"
FINAL_POS_DIR = FINAL_GEN_DIR / "positive"

POSITIVE_PROMPT = (
    "grayscale MLO mammogram, breast cancer positive, malignant finding, "
    "suspicious lesion, medical imaging"
)
NEGATIVE_PROMPT = (
    "grayscale MLO mammogram, breast cancer negative, no malignant finding, "
    "normal screening mammogram, medical imaging"
)

# Fine-tuning
RESOLUTION = 512

# Evaluation and generation
N_EVAL_IMAGES_PER_CLASS = 100
N_VALIDATION_IMAGES_PER_CLASS = 73
EVAL_INFERENCE_STEPS = 50
EVAL_GUIDANCE_SCALE = 7.5
EVAL_SEED = 42
N_FINAL_IMAGES_PER_CLASS = 1361
FINAL_GENERATE_CLASSES = ["positive", "negative"]

for directory in [
    ARCHIVES_DIR,
    DATA_AUG,
    EXPERIMENT_DIR,
    PRETRAINED_MODEL_ZIP_PATH.parent,
    HF_CACHE_DIR,
    FINAL_NEG_DIR,
    FINAL_POS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

def label_to_prompt(label):
    prompts = {0: NEGATIVE_PROMPT, 1: POSITIVE_PROMPT}
    try:
        return prompts[int(label)]
    except KeyError as exc:
        raise ValueError(f"Invalid label: {label}") from exc

print("Experiment:", EXPERIMENT_NAME)
print("Experiment directory:", EXPERIMENT_DIR)

## 2. Prepare and audit the data inputs

The data utilities accept existing processed and augmented datasets only when their expected image folders and manifests are non-empty. Otherwise, the configured ZIP archives are downloaded, validated, extracted in temporary directories, and copied into the repository. Training metadata must contain `file_name` and `label`; labels are mapped to the two fixed text prompts, and image paths are resolved according to their recorded source.

Path resolution rejects symbolic links and requires a supported image extension. A helper can copy every resolved sample into an isolated Hugging Face `imagefolder` staging directory and verify that the staged image count matches the manifest; it is defined for protocol consistency but is not invoked in this notebook because training is skipped.

The following execution cell prepares both datasets, loads `metadata_df`, prints label and source counts, and displays a seeded sample. **Inputs:** local data or Drive archives. **In-memory outputs:** validated metadata with prompt text and resolvable image paths. Invalid ZIPs, incomplete directory layouts, missing manifest columns, or unresolved files stop execution.


In [ ]:
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
EXPECTED_SPLITS = ("train", "val", "test")
EXPECTED_LABELS = ("0", "1")

def count_images(directory):
    directory = Path(directory)
    return sum(
        path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
        for path in directory.rglob("*")
    ) if directory.is_dir() else 0

def download_zip(drive_id, destination, force=False):
    destination = Path(destination)
    if destination.exists() and not force and zipfile.is_zipfile(destination):
        print("Archive already present:", destination)
        return
    destination.unlink(missing_ok=True)
    gdown.download(id=drive_id, output=str(destination), quiet=False)
    if not destination.exists() or not zipfile.is_zipfile(destination):
        raise RuntimeError(f"Invalid download: {destination}")

def processed_dataset_ready(directory):
    directory = Path(directory)
    return all(
        count_images(directory / split / label) > 0
        for split in EXPECTED_SPLITS
        for label in EXPECTED_LABELS
    )

def find_directory(root, predicate, description):
    root = Path(root)
    for candidate in [root, *(path for path in root.rglob("*") if path.is_dir())]:
        if predicate(candidate):
            return candidate
    raise FileNotFoundError(f"Directory {description} not found under {root}")

def prepare_processed_dataset():
    if processed_dataset_ready(DATA_PROCESSED_DIR) and not FORCE_PROCESSED_REDOWNLOAD:
        print("Processed dataset already ready.")
        return

    download_zip(PROCESSED_DRIVE_ID, PROCESSED_ZIP_PATH, FORCE_PROCESSED_REDOWNLOAD)
    with TemporaryDirectory(prefix="mammo_processed_extract_") as tmp:
        with zipfile.ZipFile(PROCESSED_ZIP_PATH) as archive:
            archive.extractall(tmp)
        source_dir = find_directory(tmp, processed_dataset_ready, "processed")
        shutil.copytree(source_dir, DATA_PROCESSED_DIR, dirs_exist_ok=True)

    if not processed_dataset_ready(DATA_PROCESSED_DIR):
        raise FileNotFoundError("Processed dataset is incomplete after extraction.")

def augmented_dataset_ready():
    return (DATA_AUG / "metadata.csv").is_file() and count_images(DATA_AUG) > 0

def prepare_augmented_dataset():
    if augmented_dataset_ready() and not FORCE_AUGMENTED_REDOWNLOAD:
        print("Augmented dataset already ready.")
        return

    download_zip(AUGMENTED_DRIVE_ID, AUGMENTED_ZIP_PATH, FORCE_AUGMENTED_REDOWNLOAD)
    with TemporaryDirectory(prefix="mammo_augmented_extract_") as tmp:
        with zipfile.ZipFile(AUGMENTED_ZIP_PATH) as archive:
            archive.extractall(tmp)
        source_dir = find_directory(
            tmp,
            lambda path: (path / "metadata.csv").is_file() and count_images(path) > 0,
            "augmented con metadata.csv",
        )
        DATA_AUG.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source_dir / "metadata.csv", DATA_AUG / "metadata.csv")
        for image_path in source_dir.rglob("*"):
            if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS:
                destination = DATA_AUG / image_path.name
                if not destination.exists():
                    shutil.copy2(image_path, destination)

    if not augmented_dataset_ready():
        raise FileNotFoundError("Augmented dataset is incomplete after extraction.")

def load_training_metadata():
    metadata_path = DATA_AUG / "metadata.csv"
    metadata = pd.read_csv(metadata_path).copy()
    required = {"file_name", "label"}
    missing = required.difference(metadata.columns)
    if missing:
        raise ValueError(f"Missing columns in {metadata_path}: {sorted(missing)}")
    metadata["file_name"] = metadata["file_name"].astype(str).str.replace("\\", "/", regex=False)
    metadata["label"] = metadata["label"].astype(int)
    metadata["text"] = metadata["label"].map(label_to_prompt)
    return metadata

def is_valid_training_image(path):
    path = Path(path)
    return (
        path.is_file()
        and not path.is_symlink()
        and path.suffix.lower() in IMAGE_EXTENSIONS
    )

def resolve_training_image_path(row):
    file_name = Path(str(row["file_name"]))
    label = str(int(row["label"]))
    source = str(row.get("source", "")).strip().lower()
    real_candidate = DATA_PROCESSED_DIR / "train" / label / file_name.name
    augmented_candidate = DATA_AUG / file_name.name

    if source == "real":
        candidates = [real_candidate]
    elif source in {"positive_augmentation", "augmentation", "augmented"}:
        candidates = [augmented_candidate]
    else:
        candidates = [augmented_candidate, real_candidate, PROJECT_ROOT / file_name]

    original_value = row.get("original_processed_path")
    if source == "real" and pd.notna(original_value) and str(original_value).strip():
        original = Path(str(original_value))
        if "data" in original.parts:
            candidates.append(PROJECT_ROOT / Path(*original.parts[original.parts.index("data"):]))
        candidates.append(original)

    for candidate in candidates:
        if is_valid_training_image(candidate):
            return candidate.resolve()

    checked = ", ".join(str(path) for path in candidates)
    raise FileNotFoundError(
        f"No valid image found for file_name={row['file_name']}, source={source}. "
        f"Checked paths: {checked}"
    )



In [ ]:
prepare_processed_dataset()
prepare_augmented_dataset()
metadata_df = load_training_metadata()

print("Training samples:", len(metadata_df))
print("\nDistribuzione label:")
print(metadata_df["label"].value_counts().sort_index())
if "source" in metadata_df.columns:
    print("\nDistribuzione source:")
    print(metadata_df["source"].value_counts())

sample_df = metadata_df.sample(min(6, len(metadata_df)), random_state=42)
_, axes = plt.subplots(2, 3, figsize=(12, 8))
for axis, (_, row) in zip(axes.flat, sample_df.iterrows()):
    image_path = resolve_training_image_path(row)
    with Image.open(image_path) as image:
        axis.imshow(image.convert("L"), cmap="gray")
    axis.set_title(f'label={row["label"]} | source={row.get("source", "n/a")}')
    axis.axis("off")
for axis in axes.flat[len(sample_df):]:
    axis.axis("off")
plt.tight_layout()
plt.show()

## 3. Verify the shared SD2.1 base model and local Diffusers assets

The model-preparation code requires the standard Diffusers component structure and concrete `model.safetensors` or `diffusion_pytorch_model.safetensors` files for the text encoder, U-Net, and VAE. If the shared base is already complete, it is reused. Otherwise, the configured archive is downloaded and extracted, any existing target is replaced, and physical copies are created from the FP16 filenames when standard aliases are absent; symbolic-link aliases are removed before copying.

The output is `LOCAL_MODEL_DIR`, pointing to the verified shared SD2.1 base. A second check confirms that the pinned Diffusers training script exists and prints the exact revision and script path. Missing source weights, an incomplete model structure, an invalid archive, or a missing script raises an error. These checks separate immutable base-model assets from the experiment-specific U-Net checkpoints used later and make the model-loading metadata explicit.


In [ ]:
MODEL_WEIGHT_ALIASES = {
    "text_encoder": ("model.fp16.safetensors", "model.safetensors"),
    "unet": ("diffusion_pytorch_model.fp16.safetensors", "diffusion_pytorch_model.safetensors"),
    "vae": ("diffusion_pytorch_model.fp16.safetensors", "diffusion_pytorch_model.safetensors"),
}

def has_diffusers_structure(model_dir):
    model_dir = Path(model_dir)
    required = ("model_index.json", "scheduler", "tokenizer", "text_encoder", "vae", "unet")
    return all((model_dir / item).exists() for item in required)

def local_sd_model_ready(model_dir):
    model_dir = Path(model_dir)
    weights = (
        model_dir / "text_encoder" / "model.safetensors",
        model_dir / "unet" / "diffusion_pytorch_model.safetensors",
        model_dir / "vae" / "diffusion_pytorch_model.safetensors",
    )
    return has_diffusers_structure(model_dir) and all(path.is_file() for path in weights)

def create_model_weight_copies(model_dir):
    """Always create physical copies with the standard filenames expected by Diffusers."""
    model_dir = Path(model_dir)
    for subfolder, (source_name, target_name) in MODEL_WEIGHT_ALIASES.items():
        source_path = model_dir / subfolder / source_name
        target_path = model_dir / subfolder / target_name
        if target_path.is_symlink():
            target_path.unlink()
        elif target_path.exists():
            continue
        if not source_path.is_file():
            raise FileNotFoundError(f"Source weight not found: {source_path}")
        shutil.copy2(source_path, target_path)

def prepare_pretrained_model():
    if local_sd_model_ready(PRETRAINED_MODEL_DIR) and not FORCE_MODEL_REDOWNLOAD:
        print("Stable Diffusion 2.1 model already ready.")
        return PRETRAINED_MODEL_DIR.absolute()

    download_zip(SD21_MODEL_DRIVE_ID, PRETRAINED_MODEL_ZIP_PATH, FORCE_MODEL_REDOWNLOAD)
    with TemporaryDirectory(prefix="mammo_sd21_extract_") as tmp:
        with zipfile.ZipFile(PRETRAINED_MODEL_ZIP_PATH) as archive:
            archive.extractall(tmp)
        source_dir = find_directory(tmp, has_diffusers_structure, "Diffusers model")
        if PRETRAINED_MODEL_DIR.is_symlink() or PRETRAINED_MODEL_DIR.is_file():
            PRETRAINED_MODEL_DIR.unlink()
        elif PRETRAINED_MODEL_DIR.exists():
            shutil.rmtree(PRETRAINED_MODEL_DIR)
        shutil.copytree(source_dir, PRETRAINED_MODEL_DIR)
        create_model_weight_copies(PRETRAINED_MODEL_DIR)

    if not local_sd_model_ready(PRETRAINED_MODEL_DIR):
        raise FileNotFoundError("Stable Diffusion 2.1 model is incomplete after extraction.")
    return PRETRAINED_MODEL_DIR.absolute()

LOCAL_MODEL_DIR = prepare_pretrained_model()
print("Local model:", LOCAL_MODEL_DIR)

In [ ]:
# Verify the local Diffusers repository and shared references
if not TRAIN_SCRIPT.is_file():
    raise FileNotFoundError(f"Training script not found: {TRAIN_SCRIPT}")

print("Diffusers revision:", DIFFUSERS_REVISION)
print("Script training:", TRAIN_SCRIPT)

## 4. Require the checkpoint family produced by notebook 02

No optimization is performed in G01. The following guard scans `experiments/diffusers/02_sd21_filtered_100steps/model/` for at least one `checkpoint-*` directory containing a U-Net subdirectory. If none is present, execution stops with an instruction to run notebook 02 first; otherwise, the shared candidate-checkpoint root is reported.

This dependency holds the trained checkpoint family fixed across the two notebooks, but G01 still applies its own validation-only selection under 50-step sampling. The selected weight is therefore not assumed to equal G02's 100-step selection. The cell creates no checkpoint copy and does not mutate the source experiment.


In [ ]:
# No second training run: 01 is a sampling ablation of 02.
if not any((path / 'unet').is_dir() for path in SD_OUTPUT_DIR.glob('checkpoint-*')):
    raise FileNotFoundError(
        f"Shared checkpoints not found: {SD_OUTPUT_DIR}. "
        "Run 02_SD21_Filtered_100steps.ipynb first."
    )
print("Checkpoints shared by 02:", SD_OUTPUT_DIR)

## 5. Select the checkpoint using validation data only

This section creates experiment-specific evaluation directories and verifies the validation manifest. Real validation images are resolved from metadata, sampled reproducibly by class, and copied into temporary directories that are deleted after metric computation.

All valid full-U-Net checkpoints are discovered by numeric step. For each checkpoint, the generation scheduler completes 100 negative and 100 positive samples at 50 inference steps and guidance scale 7.5. FID and Inception Score are then computed separately against up to 73 real validation images per class. Cache reuse requires an identical evaluation configuration, an unchanged validation-manifest signature, complete generated-image counts, and matching per-file signatures; unreadable or stale caches are recomputed. Sustainability information is appended only when metric computation actually runs.

The metrics are written to `checkpoint_validation_metrics.json` and the cache to `checkpoint_validation_cache.json`. The next cell reloads the metrics, orders checkpoints by ascending mean class FID, and sets `BEST_CHECKPOINT` to the first row; Inception Score is reported but is not the selection key. This 50-step selection is performed independently of G02's 100-step validation selection and can legitimately choose a different candidate. The protocol keeps model selection confined to validation data and reserves the held-out test split for final classifier evaluation.


In [ ]:
# Checkpoint-selection and final-generation configuration
EVAL_DIR = EXPERIMENT_DIR / "eval_checkpoints"
VALIDATION_METADATA_PATH = DATA_PROCESSED_DIR / "metadata" / "val.csv"

EVAL_METRICS_PATH = EXPERIMENT_DIR / "checkpoint_validation_metrics.json"
EVAL_SUSTAINABILITY_LOG = EXPERIMENT_DIR / "sustainability_validation.jsonl"

FINAL_GENERATION_LOG = EXPERIMENT_DIR / "sustainability_generation.jsonl"

for _d in [EVAL_DIR, FINAL_GEN_DIR, FINAL_NEG_DIR, FINAL_POS_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

for _metadata_path in [VALIDATION_METADATA_PATH]:
    if not _metadata_path.exists():
        raise FileNotFoundError(f"Split metadata not found: {_metadata_path}")

print(f"Source checkpoint       : {SD_OUTPUT_DIR}")
print(f"Validation metadata       : {VALIDATION_METADATA_PATH}")
print(f"Generated images/checkpoint/class: {N_EVAL_IMAGES_PER_CLASS}")
print(f"Validation images/class: {N_VALIDATION_IMAGES_PER_CLASS}")
print(f"Final images/class    : {N_FINAL_IMAGES_PER_CLASS}")

In [ ]:
# Temporary real validation references, always copied
def resolve_split_image_path(raw_path, split_name, label):
    original = Path(str(raw_path)).expanduser()
    candidates = [original] if original.is_absolute() else [PROJECT_ROOT / original]
    if "data" in original.parts:
        candidates.append(PROJECT_ROOT / Path(*original.parts[original.parts.index("data"):]))
    candidates.append(DATA_PROCESSED_DIR / split_name / str(int(label)) / original.name)

    for candidate in dict.fromkeys(candidates):
        if candidate.is_file():
            return candidate.resolve()
    checked = "\n".join(f"  - {candidate}" for candidate in dict.fromkeys(candidates))
    raise FileNotFoundError(f"Image not found for {raw_path}. Checked paths:\n{checked}")

def get_real_image_paths_from_split_metadata(metadata_path, label, n_images, seed=42):
    metadata_path = Path(metadata_path)
    metadata = pd.read_csv(metadata_path)
    required = {"label", "processed_path"}
    missing = required.difference(metadata.columns)
    if missing:
        raise ValueError(f"Missing columns in {metadata_path}: {sorted(missing)}")

    subset = metadata[metadata["label"].astype(int) == int(label)]
    if subset.empty:
        raise ValueError(f"No image with label={label} in {metadata_path}")
    n_take = len(subset) if n_images is None else min(int(n_images), len(subset))
    if n_images is not None and n_take < n_images:
        print(f"Requested {n_images} images label={label}; {n_take} available.")

    subset = subset.sample(n=n_take, random_state=seed)
    return [
        resolve_split_image_path(raw_path, metadata_path.stem.lower(), label)
        for raw_path in subset["processed_path"]
    ]

@contextmanager
def temporary_real_reference_dir_from_split_metadata(metadata_path, label, n_images, seed=42):
    image_paths = get_real_image_paths_from_split_metadata(metadata_path, label, n_images, seed)
    with TemporaryDirectory(prefix=f"real_ref_label{label}_") as tmp:
        tmp_dir = Path(tmp)
        for index, source_path in enumerate(image_paths):
            destination = tmp_dir / f"real_{label}_{index:04d}{source_path.suffix.lower()}"
            shutil.copy2(source_path, destination)
        yield tmp_dir

@contextmanager
def temporary_real_reference_dirs_from_split_metadata(metadata_path, n_per_class, seed=42):
    with temporary_real_reference_dir_from_split_metadata(
        metadata_path, label=0, n_images=n_per_class, seed=seed
    ) as real_neg_dir, temporary_real_reference_dir_from_split_metadata(
        metadata_path, label=1, n_images=n_per_class, seed=seed + 1
    ) as real_pos_dir:
        yield real_neg_dir, real_pos_dir

print("Checkpoint selection: validation.")

In [ ]:
# Checkpoint, generation, and metric utilities
def discover_checkpoints(output_dir):
    checkpoints = []
    for path in Path(output_dir).glob("checkpoint-*"):
        match = re.fullmatch(r"checkpoint-(\d+)", path.name)
        if match and path.is_dir() and (path / "unet").is_dir():
            checkpoints.append((int(match.group(1)), path))
    return sorted(checkpoints)

def load_pipeline_from_checkpoint(checkpoint_path, base_model_dir, device="cuda"):
    unet = UNet2DConditionModel.from_pretrained(
        str(Path(checkpoint_path) / "unet"),
        torch_dtype=torch.float16,
    )
    pipeline = StableDiffusionPipeline.from_pretrained(
        str(base_model_dir),
        unet=unet,
        torch_dtype=torch.float16,
        safety_checker=None,
        requires_safety_checker=False,
    ).to(device)
    pipeline.set_progress_bar_config(disable=True)
    return pipeline

def count_pngs(directory):
    return sum(path.is_file() and not path.name.startswith(".tmp_") for path in Path(directory).glob("*.png")) if Path(directory).exists() else 0

def generate_images_to_dir(
    pipeline, prompt, out_dir, n, inference_steps, guidance_scale, seed, resolution=512
):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    n_existing = count_pngs(out_dir)
    if n_existing >= n:
        print(f"    {n_existing} images already present; skipping")
        return

    print(f"    {n_existing} already present; generating {n - n_existing} missing")
    generator = torch.Generator("cuda").manual_seed(seed + n_existing)
    for index in tqdm(range(n_existing, n), desc=f"  Generation {out_dir.name}", unit="img"):
        image = pipeline(
            prompt,
            num_inference_steps=inference_steps,
            guidance_scale=guidance_scale,
            height=resolution,
            width=resolution,
            generator=generator,
        ).images[0]
        image.save(out_dir / f"gen_{index:04d}.png")

def eval_one_checkpoint(
    step,
    ckpt_path,
    base_model_dir,
    real_neg_dir,
    real_pos_dir,
    eval_dir,
    n_gen,
    inference_steps,
    guidance_scale,
    seed,
    resolution=512,
):
    class_config = {
        "negative": (NEGATIVE_PROMPT, Path(real_neg_dir), seed),
        "positive": (POSITIVE_PROMPT, Path(real_pos_dir), seed + 1),
    }
    generated_dirs = {
        name: Path(eval_dir) / f"checkpoint-{step}" / name
        for name in class_config
    }

    print(f"\nCheckpoint {step} | {Path(ckpt_path).name}")
    needs_generation = any(count_pngs(path) < n_gen for path in generated_dirs.values())
    pipeline = load_pipeline_from_checkpoint(ckpt_path, base_model_dir) if needs_generation else None
    try:
        for name, (prompt, _, class_seed) in class_config.items():
            if count_pngs(generated_dirs[name]) < n_gen:
                print(f"  Generating {name}...")
                generate_images_to_dir(
                    pipeline,
                    prompt,
                    generated_dirs[name],
                    n_gen,
                    inference_steps,
                    guidance_scale,
                    class_seed,
                    resolution,
                )
    finally:
        if pipeline is not None:
            del pipeline
            gc.collect()
            torch.cuda.empty_cache()

    metrics = {}
    for name, (_, real_dir, _) in class_config.items():
        print(f"  FID + IS: {name}")
        metrics[name] = GenerativeEvaluator(
            real_dir=real_dir,
            generated_dir=generated_dirs[name],
            batch_size=8,
            num_workers=0,
        ).compute()

    averages = {
        metric: round(sum(metrics[name][metric] for name in metrics) / len(metrics), 4)
        for metric in ("FID", "IS_mean", "IS_std")
    }
    print(f"  Avg FID={averages['FID']:.4f} | IS={averages['IS_mean']:.4f}±{averages['IS_std']:.4f}")
    return {
        "step": step,
        "ckpt_name": Path(ckpt_path).name,
        **metrics,
        "avg_FID": averages["FID"],
        "avg_IS_mean": averages["IS_mean"],
        "avg_IS_std": averages["IS_std"],
    }

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def remove_generated_duplicates_of_reused_images(final_dir, reused_prefix):
    final_dir = Path(final_dir)
    reused_hashes = {file_sha256(path) for path in final_dir.glob(f"{reused_prefix}_*.png")}
    removed = []
    for path in sorted(final_dir.glob("gen_*.png")):
        if file_sha256(path) in reused_hashes:
            path.unlink()
            removed.append(path)
    if removed:
        print(f"  Removed {len(removed)} gen_* copies overlapping reused images.")
    return removed

def duplicate_png_groups(directory):
    by_hash = {}
    for path in sorted(Path(directory).glob("*.png")):
        if path.name.startswith(".tmp_"):
            continue
        by_hash.setdefault(file_sha256(path), []).append(path.name)
    return {digest: names for digest, names in by_hash.items() if len(names) > 1}

def file_signature(path):
    path = Path(path)
    return {
        "path": str(path),
        "size": path.stat().st_size,
        "mtime_ns": path.stat().st_mtime_ns,
    }

def png_files_signature(directory):
    return [
        {
            "name": path.name,
            "size": path.stat().st_size,
            "mtime_ns": path.stat().st_mtime_ns,
        }
        for path in sorted(Path(directory).glob("*.png"))
    ]

In [ ]:
# Evaluate every checkpoint using validation data only
checkpoints = discover_checkpoints(SD_OUTPUT_DIR)
print(f"Found {len(checkpoints)} checkpoints:")
for _step, _p in checkpoints:
    print(f"  checkpoint-{_step}")

# Workers complete generation before metrics; JSON and caches remain in the notebook process.
if RUN_EVALUATION_PHASE:
    _sd_parallel_generate_checkpoint_jobs(
        checkpoints, LOCAL_MODEL_DIR, EVAL_DIR, NEGATIVE_PROMPT, POSITIVE_PROMPT,
        N_EVAL_IMAGES_PER_CLASS, EVAL_INFERENCE_STEPS, EVAL_GUIDANCE_SCALE, EVAL_SEED, RESOLUTION,
    )

CHECKPOINT_EVAL_CACHE_PATH = EXPERIMENT_DIR / "checkpoint_validation_cache.json"
FORCE_RECOMPUTE_CHECKPOINT_EVAL = False

checkpoint_eval_config = {
    "metric_backend": "generative_evaluator_finetuning.py",
    "reference_split": "validation",
    "n_per_class": N_VALIDATION_IMAGES_PER_CLASS,
    "eval_seed": EVAL_SEED,
    "inference_steps": EVAL_INFERENCE_STEPS,
    "guidance_scale": EVAL_GUIDANCE_SCALE,
    "resolution": RESOLUTION,
    "n_gen_per_class": N_EVAL_IMAGES_PER_CLASS,
}
validation_csv_signature = file_signature(VALIDATION_METADATA_PATH)

cached_checkpoints = {}
if CHECKPOINT_EVAL_CACHE_PATH.exists() and not FORCE_RECOMPUTE_CHECKPOINT_EVAL:
    try:
        with open(CHECKPOINT_EVAL_CACHE_PATH, encoding="utf-8") as f:
            _cache_payload = json.load(f)
        if (
            _cache_payload.get("schema_version") == 1
            and _cache_payload.get("config") == checkpoint_eval_config
            and _cache_payload.get("validation_csv") == validation_csv_signature
        ):
            cached_checkpoints = _cache_payload.get("checkpoints", {})
    except Exception as exc:
        print(f"Unreadable checkpoint cache; recomputing from scratch: {exc}")

all_metrics = []
recomputed_any = False
with temporary_real_reference_dirs_from_split_metadata(
    metadata_path=VALIDATION_METADATA_PATH,
    n_per_class=N_VALIDATION_IMAGES_PER_CLASS,
    seed=EVAL_SEED,
) as (real_neg_ref_dir, real_pos_ref_dir):
    print("\nDirectory temporanee validation:")
    print("  Negative:", real_neg_ref_dir)
    print("  Positive:", real_pos_ref_dir)

    with measure_sustainability(label="checkpoint_validation_evaluation", sample_interval=0.5) as eco_eval:
        for step, ckpt_path in checkpoints:
            ckpt_name = ckpt_path.name
            neg_dir = EVAL_DIR / ckpt_name / "negative"
            pos_dir = EVAL_DIR / ckpt_name / "positive"
            current_signature = {
                "negative": png_files_signature(neg_dir),
                "positive": png_files_signature(pos_dir),
            }
            cached_entry = cached_checkpoints.get(ckpt_name)
            if (
                cached_entry is not None
                and cached_entry.get("input_signature") == current_signature
                and count_pngs(neg_dir) == N_EVAL_IMAGES_PER_CLASS
                and count_pngs(pos_dir) == N_EVAL_IMAGES_PER_CLASS
            ):
                print(f"\nCheckpoint {step} | {ckpt_name}: metriche da cache, skip")
                all_metrics.append(cached_entry["metrics"])
                continue

            if not RUN_EVALUATION_PHASE:
                raise RuntimeError(
                    "Evaluation is disabled, but the compatible checkpoint-metrics cache is incomplete."
                )
            recomputed_any = True
            metrics_entry = eval_one_checkpoint(
                step=step,
                ckpt_path=ckpt_path,
                base_model_dir=LOCAL_MODEL_DIR,
                real_neg_dir=real_neg_ref_dir,
                real_pos_dir=real_pos_ref_dir,
                eval_dir=EVAL_DIR,
                n_gen=N_EVAL_IMAGES_PER_CLASS,
                inference_steps=EVAL_INFERENCE_STEPS,
                guidance_scale=EVAL_GUIDANCE_SCALE,
                seed=EVAL_SEED,
                resolution=RESOLUTION,
            )
            all_metrics.append(metrics_entry)
            cached_checkpoints[ckpt_name] = {
                "input_signature": {
                    "negative": png_files_signature(neg_dir),
                    "positive": png_files_signature(pos_dir),
                },
                "metrics": metrics_entry,
            }

with open(EVAL_METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(all_metrics, f, indent=2, ensure_ascii=False)

with open(CHECKPOINT_EVAL_CACHE_PATH, "w", encoding="utf-8") as f:
    json.dump({
        "schema_version": 1,
        "config": checkpoint_eval_config,
        "validation_csv": validation_csv_signature,
        "checkpoints": cached_checkpoints,
    }, f, indent=2, ensure_ascii=False)

if recomputed_any:
    _eco_record = eco_eval.metrics.to_dict()
    _eco_record.update({
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "record_type": "checkpoint_validation_evaluation",
        "n_checkpoints": len(checkpoints),
        "n_real_images_per_class": N_VALIDATION_IMAGES_PER_CLASS,
        "n_generated_images_per_class": N_EVAL_IMAGES_PER_CLASS,
        "real_reference_metadata": str(VALIDATION_METADATA_PATH),
    })
    with open(EVAL_SUSTAINABILITY_LOG, "a", encoding="utf-8") as f:
        f.write(json.dumps(_eco_record, ensure_ascii=False) + "\n")
    print(f"\nMetriche eco-tracking validation:\n{eco_eval.metrics}")
else:
    print("\nAll metrics loaded from cache; no checkpoint was recomputed.")

print(f"Metriche validation salvate in: {EVAL_METRICS_PATH}")

In [ ]:
# Select the best checkpoint
# Reload from file (idempotent when the cell is rerun)
with open(EVAL_METRICS_PATH, encoding="utf-8") as f:
    all_metrics = json.load(f)

df_eval = pd.DataFrame([
    {
        "checkpoint":  m["ckpt_name"],
        "step":        m["step"],
        "avg_FID":     m["avg_FID"],
        "avg_IS_mean": m["avg_IS_mean"],
        "avg_IS_std":  m["avg_IS_std"],
        "FID_neg":     m["negative"]["FID"],
        "FID_pos":     m["positive"]["FID"],
        "IS_neg":      m["negative"]["IS_mean"],
        "IS_pos":      m["positive"]["IS_mean"],
    }
    for m in all_metrics
]).sort_values("avg_FID").reset_index(drop=True)

print("Metric summary (sorted by increasing avg_FID, best first):")
print(df_eval.to_string(index=False))

best_row        = df_eval.iloc[0]
BEST_CHECKPOINT = SD_OUTPUT_DIR / best_row["checkpoint"]

print(f"\n✓ Best checkpoint : {best_row['checkpoint']}")
print(f"  avg FID  : {best_row['avg_FID']:.4f}")
print(f"  avg IS   : {best_row['avg_IS_mean']:.4f} ± {best_row['avg_IS_std']:.4f}")
print(f"  Path     : {BEST_CHECKPOINT}")

## 6. Construct the final unfiltered synthetic image pools

The selected checkpoint defines a target of 1,361 PNGs per class. Evaluation images are copied into the final directories during a generation or repair pass, and isolated workers fill only missing deterministic indices. A complete local pool is reused when the final generation plan confirms readable expected filenames, the exact target count, and unique image content. The per-directory generation manifest is validated only when missing images must be scheduled; historical manifest fingerprints do not gate reuse of an already complete pool.

When `generation_info.json` exists, a selected-checkpoint mismatch stops execution. During an active generation pass, content duplicates between reused evaluation images and newly generated images are removed; duplicate groups remain fatal in every path. GPU memory is released after the final verification.

**Inputs:** `BEST_CHECKPOINT`, the shared base model, class prompts, and the selected checkpoint's evaluation images. **Outputs:** unfiltered negative and positive pools beneath `generated_images/final/`, plus `generation_info.json` and an append-only sustainability log. The record includes checkpoint identity, validation-selection metrics, image counts, sampling parameters, seed strategy, and generation-device policy. This stage materializes the G01 50-step dataset; it does not apply the adaptive filter used by other experiments. Whether it forms a sampling-step-only ablation with G02 is established later from saved checkpoint identity, not assumed here.


In [ ]:
# Final generation with the best checkpoint
print(f"Best checkpoint  : {BEST_CHECKPOINT.name}")
print(f"Images/class  : {N_FINAL_IMAGES_PER_CLASS}")
print(f"Output           : {FINAL_GEN_DIR}")

BEST_EVAL_DIR = EVAL_DIR / BEST_CHECKPOINT.name
_class_config = {
    "negative": {
        "prompt": NEGATIVE_PROMPT,
        "final_dir": FINAL_NEG_DIR,
        "eval_dir": BEST_EVAL_DIR / "negative",
        "seed": EVAL_SEED,
        "prefix": "eval_neg",
    },
    "positive": {
        "prompt": POSITIVE_PROMPT,
        "final_dir": FINAL_POS_DIR,
        "eval_dir": BEST_EVAL_DIR / "positive",
        "seed": EVAL_SEED + 1,
        "prefix": "eval_pos",
    },
}

from parallel_generation_utils import SD_SEED_STRATEGY, copy_validated_sd_evaluation_images, final_sd_generation_plan

def _final_plan(cfg):
    return final_sd_generation_plan(Path(cfg["final_dir"]), N_FINAL_IMAGES_PER_CLASS, cfg["prefix"])

_final_already_complete = all(
    _final_plan(_class_config[_cls])["complete"]
    for _cls in FINAL_GENERATE_CLASSES
)
if _final_already_complete:
    print("Final PNG set complete; still validating files and manifest parameters before skipping.")
generation_info_path = EXPERIMENT_DIR / "generation_info.json"
if generation_info_path.exists():
    with open(generation_info_path, encoding="utf-8") as f:
        previous_generation = json.load(f)
    previous_best = previous_generation.get("best_checkpoint")
    final_images_exist = any(any(Path(cfg["final_dir"]).glob("*.png")) for cfg in _class_config.values())
    if previous_best and previous_best != BEST_CHECKPOINT.name and final_images_exist:
        raise RuntimeError(
            f"The best checkpoint changed from {previous_best} to {BEST_CHECKPOINT.name}. "
            "Use a new FINAL_GEN_DIR or deliberately clear the final directories."
        )

removed_overlaps = {}

with measure_sustainability(label=f"final_gen_{BEST_CHECKPOINT.name}", sample_interval=0.5) as eco_final:
    for _cls in FINAL_GENERATE_CLASSES:
        cfg = _class_config[_cls]
        eval_available = N_EVAL_IMAGES_PER_CLASS

        print(f"\nClass {_cls.upper()}")
        print(f"  Final target              : {N_FINAL_IMAGES_PER_CLASS}")
        print(f"  Reused evaluation images : {eval_available}")

        if RUN_GENERATION_PHASE:
            # Copy reused images first, then complete the total
            reuse_record = copy_validated_sd_evaluation_images(
                source_dir=cfg["eval_dir"], final_dir=cfg["final_dir"], count=eval_available,
                reused_prefix=cfg["prefix"], checkpoint_path=str(BEST_CHECKPOINT),
                class_name=_cls, prompt=cfg["prompt"],
                base_seed=EVAL_SEED, num_inference_steps=EVAL_INFERENCE_STEPS,
                guidance_scale=EVAL_GUIDANCE_SCALE, resolution=RESOLUTION,
                checkpoint_type=SD_CHECKPOINT_TYPE, base_model_dir=LOCAL_MODEL_DIR,
            )
            n_reused = len(reuse_record["files"])
            n_new_target = N_FINAL_IMAGES_PER_CLASS - n_reused
            print(f"  Actually copied from evaluation: {n_reused}")
            print(f"  gen_* share of the final dataset : {n_new_target}")

        else:
            _existing_plan = _final_plan(cfg)
            if not _existing_plan["complete"]:
                raise RuntimeError(
                    f"Generation is disabled, but the existing {_cls} pool is incomplete: {_existing_plan}"
                )
            n_reused = _existing_plan["n_valid_reused"]
            n_new_target = N_FINAL_IMAGES_PER_CLASS - n_reused
            print(f"  Existing complete pool validated; reused-prefix images: {n_reused}")

        if RUN_GENERATION_PHASE:
            removed = remove_generated_duplicates_of_reused_images(
                final_dir=cfg["final_dir"],
                reused_prefix=cfg["prefix"],
            )
            removed_overlaps[_cls] = len(removed)

        else:
            removed = []

        _plan_before = _final_plan(cfg)
        n_before = _plan_before["n_valid_reused"] + len(_plan_before["valid_gen_indices"])
        print(f"  Unique images present   : {n_before}")
        print(f"  New images to generate          : {max(0, N_FINAL_IMAGES_PER_CLASS - n_before)}")

        if RUN_GENERATION_PHASE:
            _sd_parallel_generate_final(
                BEST_CHECKPOINT, LOCAL_MODEL_DIR, cfg["prompt"], cfg["final_dir"], N_FINAL_IMAGES_PER_CLASS,
                EVAL_INFERENCE_STEPS, EVAL_GUIDANCE_SCALE, cfg["seed"], RESOLUTION, _cls, cfg["prefix"], n_reused,
            )

        duplicates = duplicate_png_groups(cfg["final_dir"])
        if duplicates:
            raise RuntimeError(f"Found {len(duplicates)} duplicate groups under {cfg['final_dir']}.")
        _verified_plan = _final_plan(cfg)
        if not _verified_plan["complete"]:
            raise RuntimeError(f"Invalid final dataset per {_cls}: {_verified_plan}")

gc.collect()
torch.cuda.empty_cache()

_n_neg = N_FINAL_IMAGES_PER_CLASS if _final_plan(_class_config["negative"])["complete"] else 0
_n_pos = N_FINAL_IMAGES_PER_CLASS if _final_plan(_class_config["positive"])["complete"] else 0
_final_record = eco_final.metrics.to_dict()
_final_record.update({
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "record_type": "final_generation",
    "generation_log_schema": 2,
    "best_checkpoint": BEST_CHECKPOINT.name,
    "selection_reference": "validation",
    "selection_metrics_path": str(EVAL_METRICS_PATH),
    "avg_FID": float(best_row["avg_FID"]),
    "avg_IS_mean": float(best_row["avg_IS_mean"]),
    "n_per_class": N_FINAL_IMAGES_PER_CLASS,
    "n_eval_reused": N_EVAL_IMAGES_PER_CLASS,
    "removed_overlaps": removed_overlaps,
    "n_negative_final": _n_neg,
    "n_positive_final": _n_pos,
    "inference_steps": EVAL_INFERENCE_STEPS,
    "guidance_scale": EVAL_GUIDANCE_SCALE,
    "generated_classes": FINAL_GENERATE_CLASSES,
    "generated_negative": str(FINAL_NEG_DIR),
    "generated_positive": str(FINAL_POS_DIR),
    "status": "completed",
    "seed_strategy": SD_SEED_STRATEGY,
    "parallel_generation": PARALLEL_GENERATION,
    "generation_gpu_devices": GENERATION_GPU_DEVICES,
})

if RUN_GENERATION_PHASE:
    with open(generation_info_path, "w", encoding="utf-8") as f:
        json.dump(_final_record, f, indent=2, ensure_ascii=False)
    with open(FINAL_GENERATION_LOG, "a", encoding="utf-8") as f:
        f.write(json.dumps(_final_record, ensure_ascii=False) + "\n")
else:
    print("Generation phase skipped; existing pool and checkpoint compatibility were validated without mutation.")

print(f"\nFinal-generation EcoTracker metrics:\n{eco_final.metrics}")
print(f"Positive finali: {_n_pos} -> {FINAL_POS_DIR}")